# Clases y objetos · TAD de pilas y colas

Un **tipo abstracto de datos** (TAD) se define por sus operaciones y por las propiedades que cumplen, no por cómo se guardan los datos. Aquí cada TAD es una clase abstracta con unas pocas operaciones primitivas (para la pila: `vacia`, `apilar`, `cima`, `desapilar` y `es_vacia`), tiene varias implementaciones intercambiables y el resto de funciones (conversiones, filtrado, pertenencia, subpilas, ordenación, intercalado de colas…) se escriben usando **solo** esa interfaz, así que valen para todas sin cambiar una línea. Las implementaciones son persistentes: cada operación devuelve una estructura nueva y deja intacta la original. Sobre `list` y `deque` eso obliga a copiar ($O(n)$ por operación), mientras que la cola de dos listas enlazadas comparte estructura y consigue $O(1)$ amortizado. Las propiedades de cada TAD y la equivalencia entre implementaciones se comprueban con [Hypothesis](https://hypothesis.readthedocs.io/), con un perfil reproducible (`derandomize=True`, sin base de datos de fallos).

*Estructuras lineales* implementa pilas y colas mutables y las aplica (paréntesis, notación polaca); aquí el foco es la separación entre interfaz e implementación. Ideas de los capítulos 6 a 8 del libro [*Ejercicios de programación con Python*](https://github.com/jaalonso/Ejercicios-Python) de José A. Alonso Jiménez.

In [1]:
import bisect
import heapq
import random
from abc import ABC, abstractmethod
from collections import deque
from functools import reduce
from itertools import zip_longest
from typing import Self

from hypothesis import given, settings
from hypothesis import strategies as st

settings.register_profile("cuaderno", derandomize=True, database=None, deadline=None)
settings.load_profile("cuaderno")


def comprobar(prueba, nombre=None):
    """Ejecuta una prueba de Hypothesis; si falla, muestra el contraejemplo reducido y devuelve el error."""
    nombre = nombre or prueba.__name__
    try:
        prueba()
    except Exception as e:
        print(f"{nombre}: FALLA ({type(e).__name__})")
        for nota in getattr(e, "__notes__", []):
            # La explicación que sigue incluye rutas de ficheros locales
            if nota.startswith("Explanation"):
                break
            print("   ", nota.replace("\n", "\n    "))
        return e
    print(f"{nombre}: pasa")

## 1. La interfaz del TAD pila

Definir la clase abstracta genérica `Pila[T]` con las cinco operaciones primitivas como métodos abstractos: `vacia()` (método de clase), `apilar(x)`, `cima()`, `desapilar()` y `es_vacia()`. Ninguna modifica la pila: `apilar` y `desapilar` devuelven una pila nueva. Escribir en la propia clase, usando solo esas operaciones, `__str__` (de la cima al fondo, como `5 | 2 | 3 | -`), `__repr__` y `__eq__`, que así sirven para cualquier implementación, incluso para comparar dos implementaciones distintas. Comprobar que no se puede crear una pila de una subclase a la que le falten operaciones.

Toda implementación debe cumplir: $\text{cima}(\text{apilar}(x, p)) = x$, $\text{desapilar}(\text{apilar}(x, p)) = p$, $\text{vacia}$ es vacía y $\text{apilar}(x, p)$ no lo es. Un `typing.Protocol` describiría la misma interfaz sin herencia, pero cada implementación tendría que repetir `__str__` y `__eq__`, y nada impediría instanciar una incompleta.

In [2]:
class Pila[T](ABC):
    """TAD pila: solo los métodos abstractos conocen la representación."""

    __slots__ = ()

    @classmethod
    @abstractmethod
    def vacia(cls) -> Self: ...

    @abstractmethod
    def apilar(self, x: T) -> Self: ...

    @abstractmethod
    def cima(self) -> T: ...

    @abstractmethod
    def desapilar(self) -> Self: ...

    @abstractmethod
    def es_vacia(self) -> bool: ...

    def __str__(self):
        partes, p = [], self
        while not p.es_vacia():
            partes.append(str(p.cima()))
            p = p.desapilar()
        return " | ".join(partes + ["-"])

    def __repr__(self):
        return f"{type(self).__name__}({self})"

    def __eq__(self, otra):
        if not isinstance(otra, Pila):
            return NotImplemented
        p, q = self, otra
        while not (p.es_vacia() or q.es_vacia()):
            if p.cima() != q.cima():
                return False
            p, q = p.desapilar(), q.desapilar()
        return p.es_vacia() and q.es_vacia()


class PilaIncompleta(Pila):
    def apilar(self, x):
        return self


try:
    PilaIncompleta()
except TypeError as e:
    print("TypeError:", e)

TypeError: Can't instantiate abstract class PilaIncompleta without an implementation for abstract methods 'cima', 'desapilar', 'es_vacia', 'vacia'


## 2. Dos implementaciones: `list` y `deque`

Implementar `PilaLista`, que guarda los elementos en una `list` con la cima al final, y `PilaDeque`, que los guarda en un `deque` con la cima a la izquierda. Para no modificar la pila original, `apilar` y `desapilar` trabajan sobre una copia. Consultar la cima o desapilar una pila vacía lanza `IndexError`. Construir la pila 3, 2, 5 con las dos, comprobar que la original no cambia al apilar y que las dos pilas son iguales aunque su representación sea distinta. Después comprobar con Hypothesis las cuatro propiedades del TAD en ambas implementaciones.

In [3]:
class PilaLista[T](Pila[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacia(cls):
        return cls()

    def apilar(self, x):
        return type(self)(self._elementos + [x])

    def cima(self):
        if not self._elementos:
            raise IndexError("cima de una pila vacía")
        return self._elementos[-1]

    def desapilar(self):
        if not self._elementos:
            raise IndexError("desapilar una pila vacía")
        return type(self)(self._elementos[:-1])

    def es_vacia(self):
        return not self._elementos


class PilaDeque[T](Pila[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=None):
        self._elementos = deque() if elementos is None else elementos

    @classmethod
    def vacia(cls):
        return cls()

    def apilar(self, x):
        d = self._elementos.copy()
        d.appendleft(x)
        return type(self)(d)

    def cima(self):
        if not self._elementos:
            raise IndexError("cima de una pila vacía")
        return self._elementos[0]

    def desapilar(self):
        if not self._elementos:
            raise IndexError("desapilar una pila vacía")
        d = self._elementos.copy()
        d.popleft()
        return type(self)(d)

    def es_vacia(self):
        return not self._elementos


IMPLEMENTACIONES_PILA = (PilaLista, PilaDeque)

for impl in IMPLEMENTACIONES_PILA:
    p = impl.vacia().apilar(3).apilar(2).apilar(5)
    q = p.apilar(7)
    print(f"{impl.__name__:9}  p = {p}   p.apilar(7) = {q}   p sigue: {p}   interna: {p._elementos}")
    for operacion in (impl.vacia().cima, impl.vacia().desapilar):
        try:
            operacion()
        except IndexError as e:
            print(f"{'':11}IndexError: {e}")

p1 = PilaLista.vacia().apilar(3).apilar(2).apilar(5)
p2 = PilaDeque.vacia().apilar(3).apilar(2).apilar(5)
print(repr(p1), "==", repr(p2), "->", p1 == p2)
assert p1 == p2 and p1 != p2.desapilar() and p1.desapilar().desapilar() == PilaDeque.vacia().apilar(3)

PilaLista  p = 5 | 2 | 3 | -   p.apilar(7) = 7 | 5 | 2 | 3 | -   p sigue: 5 | 2 | 3 | -   interna: [3, 2, 5]
           IndexError: cima de una pila vacía
           IndexError: desapilar una pila vacía
PilaDeque  p = 5 | 2 | 3 | -   p.apilar(7) = 7 | 5 | 2 | 3 | -   p sigue: 5 | 2 | 3 | -   interna: deque([5, 2, 3])
           IndexError: cima de una pila vacía
           IndexError: desapilar una pila vacía
PilaLista(5 | 2 | 3 | -) == PilaDeque(5 | 2 | 3 | -) -> True


In [4]:
def pilas(impl, elementos=st.integers(-50, 50), max_size=20):
    """Estrategia de Hypothesis que genera pilas de la implementación dada."""
    return st.lists(elementos, max_size=max_size).map(lambda xs: reduce(lambda p, x: p.apilar(x), xs, impl.vacia()))


for impl in IMPLEMENTACIONES_PILA:

    @given(pilas(impl), st.integers())
    def axiomas_pila(p, x):
        assert p.apilar(x).cima() == x
        assert p.apilar(x).desapilar() == p
        assert impl.vacia().es_vacia()
        assert not p.apilar(x).es_vacia()

    assert comprobar(axiomas_pila, f"axiomas de {impl.__name__}") is None

axiomas de PilaLista: pasa
axiomas de PilaDeque: pasa


## 3. De lista a pila y de pila a lista

Escribir `lista_a_pila(xs, impl)`, que construya una pila de la implementación `impl` en la que el último elemento de la lista queda en la cima, y `pila_a_lista(p)`, que devuelva los elementos del fondo a la cima. Comprobar con Hypothesis, en las dos implementaciones, que son inversas: `pila_a_lista(lista_a_pila(xs))` es `xs` y `lista_a_pila(pila_a_lista(p))` es `p`.

In [5]:
def lista_a_pila(xs, impl):
    p = impl.vacia()
    for x in xs:
        p = p.apilar(x)
    return p


def pila_a_lista(p):
    xs = []
    while not p.es_vacia():
        xs.append(p.cima())
        p = p.desapilar()
    return xs[::-1]


p = lista_a_pila([3, 2, 5], PilaDeque)
print(p, "->", pila_a_lista(p))

for impl in IMPLEMENTACIONES_PILA:

    @given(st.lists(st.integers()))
    def ida_y_vuelta(xs):
        assert pila_a_lista(lista_a_pila(xs, impl)) == xs

    @given(pilas(impl))
    def vuelta_e_ida(p):
        assert lista_a_pila(pila_a_lista(p), impl) == p

    assert comprobar(ida_y_vuelta, f"lista → pila → lista ({impl.__name__})") is None
    assert comprobar(vuelta_e_ida, f"pila → lista → pila ({impl.__name__})") is None

5 | 2 | 3 | - -> [3, 2, 5]
lista → pila → lista (PilaLista): pasa
pila → lista → pila (PilaLista): pasa
lista → pila → lista (PilaDeque): pasa


pila → lista → pila (PilaDeque): pasa


## 4. Filtrar y transformar una pila

Escribir `filtrar_pila(pred, p)`, con los elementos de `p` que cumplen `pred` en el mismo orden, y `mapear_pila(f, p)`, que aplique `f` a cada elemento. Las dos se definen por recursión: se resuelve el problema para `desapilar(p)` y después se vuelve a apilar (o no) la cima. Comprobarlas contra comprensiones de listas.

In [6]:
def filtrar_pila(pred, p):
    if p.es_vacia():
        return p
    resto = filtrar_pila(pred, p.desapilar())
    return resto.apilar(p.cima()) if pred(p.cima()) else resto


def mapear_pila(f, p):
    if p.es_vacia():
        return p
    return mapear_pila(f, p.desapilar()).apilar(f(p.cima()))


p = lista_a_pila([4, 7, 10, 3, 8], PilaLista)
print("p              :", p)
print("pares          :", filtrar_pila(lambda x: x % 2 == 0, p))
print("al cuadrado    :", mapear_pila(lambda x: x * x, p))
assert pila_a_lista(filtrar_pila(lambda x: x > 5, p)) == [7, 10, 8]

for impl in IMPLEMENTACIONES_PILA:

    @given(pilas(impl))
    def filtrar_y_mapear(p):
        xs = pila_a_lista(p)
        assert pila_a_lista(filtrar_pila(lambda x: x % 3 == 0, p)) == [x for x in xs if x % 3 == 0]
        assert pila_a_lista(mapear_pila(lambda x: 2 * x + 1, p)) == [2 * x + 1 for x in xs]

    assert comprobar(filtrar_y_mapear, f"filtrar y mapear ({impl.__name__})") is None

p              : 8 | 3 | 10 | 7 | 4 | -
pares          : 8 | 10 | 4 | -
al cuadrado    : 64 | 9 | 100 | 49 | 16 | -
filtrar y mapear (PilaLista): pasa


filtrar y mapear (PilaDeque): pasa


## 5. Pertenencia y contención

Escribir `pertenece_pila(x, p)`, que diga si `x` está en la pila, y `contenida_pila(p1, p2)`, que diga si todos los elementos de `p1` están en `p2` (sin importar el orden ni las repeticiones). Las dos pilas pueden ser de implementaciones distintas. Comprobarlas contra `in` y `all` sobre listas, con valores pequeños para que haya muchos casos positivos.

In [7]:
def pertenece_pila(x, p):
    while not p.es_vacia():
        if p.cima() == x:
            return True
        p = p.desapilar()
    return False


def contenida_pila(p1, p2):
    while not p1.es_vacia():
        if not pertenece_pila(p1.cima(), p2):
            return False
        p1 = p1.desapilar()
    return True


p1 = lista_a_pila([2, 5, 2], PilaLista)
p2 = lista_a_pila([5, 3, 2, 1], PilaDeque)
print(f"{p1} contenida en {p2}: {contenida_pila(p1, p2)};  al revés: {contenida_pila(p2, p1)}")


@given(pilas(PilaLista, st.integers(0, 5), 8), pilas(PilaDeque, st.integers(0, 5), 8), st.integers(0, 5))
def pertenencia_y_contencion(p1, p2, x):
    xs, ys = pila_a_lista(p1), pila_a_lista(p2)
    assert pertenece_pila(x, p1) == (x in xs)
    assert contenida_pila(p1, p2) == all(x in ys for x in xs)


assert comprobar(pertenencia_y_contencion) is None

2 | 5 | 2 | - contenida en 1 | 2 | 3 | 5 | -: True;  al revés: False
pertenencia_y_contencion: pasa


## 6. Prefijo y subpila

Escribir `es_prefijo_pila(p1, p2)`, que diga si los elementos de `p1` aparecen, desde la cima, al principio de `p2`, y `es_subpila(p1, p2)`, que diga si aparecen en `p2` como un bloque de elementos consecutivos. Comprobarlas contra cortes de listas y, además, que cualquier tramo consecutivo de una pila es una subpila suya.

In [8]:
def es_prefijo_pila(p1, p2):
    while not p1.es_vacia():
        if p2.es_vacia() or p1.cima() != p2.cima():
            return False
        p1, p2 = p1.desapilar(), p2.desapilar()
    return True


def es_subpila(p1, p2):
    while not es_prefijo_pila(p1, p2):
        if p2.es_vacia():
            return False
        p2 = p2.desapilar()
    return True


def es_sublista(xs, ys):
    return any(ys[i : i + len(xs)] == xs for i in range(len(ys) - len(xs) + 1))


p = lista_a_pila([1, 6, 3, 9, 4], PilaLista)
for xs in ([9, 4], [6, 3, 9], [6, 9], [1, 6, 3, 9, 4, 0]):
    q = lista_a_pila(xs, PilaDeque)
    print(f"{str(q):26} prefijo: {es_prefijo_pila(q, p)!s:5}  subpila: {es_subpila(q, p)!s:5}  de {p}")


@given(pilas(PilaLista, st.integers(0, 2), 4), pilas(PilaDeque, st.integers(0, 2), 10))
def prefijo_y_subpila(p1, p2):
    # Las listas se leen desde la cima
    xs, ys = pila_a_lista(p1)[::-1], pila_a_lista(p2)[::-1]
    assert es_prefijo_pila(p1, p2) == (ys[: len(xs)] == xs)
    assert es_subpila(p1, p2) == es_sublista(xs, ys)


@given(st.lists(st.integers(), max_size=12), st.integers(0, 12), st.integers(0, 12))
def tramos_son_subpilas(ys, i, j):
    i, j = sorted((i, j))
    assert es_subpila(lista_a_pila(ys[i:j], PilaDeque), lista_a_pila(ys, PilaLista))


assert comprobar(prefijo_y_subpila) is None
assert comprobar(tramos_son_subpilas) is None

4 | 9 | -                  prefijo: True   subpila: True   de 4 | 9 | 3 | 6 | 1 | -
9 | 3 | 6 | -              prefijo: False  subpila: True   de 4 | 9 | 3 | 6 | 1 | -
9 | 6 | -                  prefijo: False  subpila: False  de 4 | 9 | 3 | 6 | 1 | -
0 | 4 | 9 | 3 | 6 | 1 | -  prefijo: False  subpila: False  de 4 | 9 | 3 | 6 | 1 | -


prefijo_y_subpila: pasa


tramos_son_subpilas: pasa


## 7. Pila ordenada y ordenación por inserción

Escribir `ordenada_pila(p)`, que diga si los elementos crecen (o se repiten) de la cima al fondo; `insertar_ordenado(x, p)`, que coloque `x` en su sitio en una pila ordenada; y `ordenar_pila(p)`, que ordene una pila por inserción: ordena el resto y después inserta la cima. Comprobar que el resultado está ordenado y coincide con `sorted`.

In [9]:
def ordenada_pila(p):
    while not p.es_vacia():
        x, p = p.cima(), p.desapilar()
        if not p.es_vacia() and x > p.cima():
            return False
    return True


def insertar_ordenado(x, p):
    if p.es_vacia() or x <= p.cima():
        return p.apilar(x)
    return insertar_ordenado(x, p.desapilar()).apilar(p.cima())


def ordenar_pila(p):
    if p.es_vacia():
        return p
    return insertar_ordenado(p.cima(), ordenar_pila(p.desapilar()))


p = lista_a_pila([4, 1, 7, 3, 7, 2], PilaDeque)
print(f"{p}  ordenada: {ordenada_pila(p)}")
print(f"{ordenar_pila(p)}  ordenada: {ordenada_pila(ordenar_pila(p))}")

for impl in IMPLEMENTACIONES_PILA:

    @given(pilas(impl, st.integers(0, 9)))
    def ordenacion(p):
        xs = pila_a_lista(p)
        assert ordenada_pila(p) == (xs[::-1] == sorted(xs))
        q = ordenar_pila(p)
        assert ordenada_pila(q) and pila_a_lista(q) == sorted(xs, reverse=True)

    assert comprobar(ordenacion, f"ordenación por inserción ({impl.__name__})") is None

2 | 7 | 3 | 7 | 1 | 4 | -  ordenada: False
1 | 2 | 3 | 4 | 7 | 7 | -  ordenada: True


ordenación por inserción (PilaLista): pasa


ordenación por inserción (PilaDeque): pasa


## 8. Quitar repetidos, invertir y máximo

Escribir `invertir_pila(p)`; `sin_repetidos_pila(p)`, que deje solo la aparición más cercana a la cima de cada elemento (llevando en una pila auxiliar los ya vistos); y `maximo_pila(p)`, que lance `ValueError` si la pila está vacía. Comprobarlas contra `reversed`, `dict.fromkeys` y `max`.

In [10]:
def invertir_pila(p):
    q = p.vacia()
    while not p.es_vacia():
        q, p = q.apilar(p.cima()), p.desapilar()
    return q


def sin_repetidos_pila(p):
    vistos = p.vacia()
    while not p.es_vacia():
        if not pertenece_pila(p.cima(), vistos):
            vistos = vistos.apilar(p.cima())
        p = p.desapilar()
    return invertir_pila(vistos)


def maximo_pila(p):
    if p.es_vacia():
        raise ValueError("máximo de una pila vacía")
    m = p.cima()
    while not (p := p.desapilar()).es_vacia():
        m = max(m, p.cima())
    return m


p = lista_a_pila([3, 5, 1, 3, 2, 5, 4], PilaLista)
print(f"p             : {p}")
print(f"invertida     : {invertir_pila(p)}")
print(f"sin repetidos : {sin_repetidos_pila(p)}")
print(f"máximo        : {maximo_pila(p)}")
try:
    maximo_pila(PilaLista.vacia())
except ValueError as e:
    print("ValueError:", e)

for impl in IMPLEMENTACIONES_PILA:

    @given(pilas(impl, st.integers(0, 6)))
    def repetidos_y_maximo(p):
        desde_cima = pila_a_lista(p)[::-1]
        assert pila_a_lista(invertir_pila(p)) == desde_cima
        assert pila_a_lista(sin_repetidos_pila(p))[::-1] == list(dict.fromkeys(desde_cima))
        if desde_cima:
            assert maximo_pila(p) == max(desde_cima)

    assert comprobar(repetidos_y_maximo, f"repetidos y máximo ({impl.__name__})") is None

p             : 4 | 5 | 2 | 3 | 1 | 5 | 3 | -
invertida     : 3 | 5 | 1 | 3 | 2 | 5 | 4 | -
sin repetidos : 4 | 5 | 2 | 3 | 1 | -
máximo        : 5
ValueError: máximo de una pila vacía
repetidos y máximo (PilaLista): pasa


repetidos y máximo (PilaDeque): pasa

## 9. La interfaz del TAD cola y una implementación con `list`

Definir la clase abstracta `Cola[T]` con `vacia()`, `encolar(x)` (añade al final), `primero()`, `resto()` (la cola sin su primer elemento) y `es_vacia()`. Escribir el generador `recorrer(c)`, que dé los elementos del primero al último usando solo `primero`, `resto` y `es_vacia`, y con él `__str__` (`3 | 2 | 5`, o `(vacía)`) y `__eq__`. Implementar `ColaLista` sobre una `list` copiada en cada operación. Toda implementación debe cumplir:

- $\text{primero}(\text{encolar}(x, \text{vacia})) = x$ y, si $c$ no es vacía, $\text{primero}(\text{encolar}(x, c)) = \text{primero}(c)$.
- $\text{resto}(\text{encolar}(x, \text{vacia})) = \text{vacia}$ y, si $c$ no es vacía, $\text{resto}(\text{encolar}(x, c)) = \text{encolar}(x, \text{resto}(c))$.
- $\text{vacia}$ es vacía y $\text{encolar}(x, c)$ no lo es.

In [11]:
_FALTA = object()


def recorrer(c):
    while not c.es_vacia():
        yield c.primero()
        c = c.resto()


class Cola[T](ABC):
    __slots__ = ()

    @classmethod
    @abstractmethod
    def vacia(cls) -> Self: ...

    @abstractmethod
    def encolar(self, x: T) -> Self: ...

    @abstractmethod
    def primero(self) -> T: ...

    @abstractmethod
    def resto(self) -> Self: ...

    @abstractmethod
    def es_vacia(self) -> bool: ...

    def __str__(self):
        return " | ".join(map(str, recorrer(self))) or "(vacía)"

    def __repr__(self):
        return f"{type(self).__name__}({self})"

    def __eq__(self, otra):
        if not isinstance(otra, Cola):
            return NotImplemented
        return all(a == b for a, b in zip_longest(recorrer(self), recorrer(otra), fillvalue=_FALTA))


class ColaLista[T](Cola[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacia(cls):
        return cls()

    def encolar(self, x):
        return type(self)(self._elementos + [x])

    def primero(self):
        if not self._elementos:
            raise IndexError("primero de una cola vacía")
        return self._elementos[0]

    def resto(self):
        if not self._elementos:
            raise IndexError("resto de una cola vacía")
        return type(self)(self._elementos[1:])

    def es_vacia(self):
        return not self._elementos


c = ColaLista.vacia().encolar(3).encolar(2).encolar(5)
print(f"c = {c};  primero: {c.primero()};  resto: {c.resto()};  c.encolar(8) = {c.encolar(8)};  c sigue: {c}")
print(repr(ColaLista.vacia()))
assert c.resto().resto() == ColaLista.vacia().encolar(5) and c != c.resto()

c = 3 | 2 | 5;  primero: 3;  resto: 2 | 5;  c.encolar(8) = 3 | 2 | 5 | 8;  c sigue: 3 | 2 | 5
ColaLista((vacía))


## 10. Cola con dos listas enlazadas y cola con `deque`

Implementar `ColaDosListas` con dos listas enlazadas inmutables (pares `(cabeza, cola)` terminados en `None`): `frente` con los primeros elementos en orden y `detras` con los últimos al revés, de modo que encolar es añadir una cabeza a `detras`, $O(1)$. Mantener el invariante «si `frente` está vacía, `detras` también»: cuando `frente` se queda vacía se vuelca en ella `detras` invertida. Implementar también `ColaDeque` sobre un `deque` copiado. Comprobar con Hypothesis las propiedades del TAD en las tres implementaciones, generando colas con secuencias mezcladas de `encolar` y `resto` para que `ColaDosListas` tenga repartos distintos entre sus dos listas.

In [12]:
class ColaDosListas[T](Cola[T]):
    __slots__ = ("_frente", "_detras")

    def __init__(self, frente=None, detras=None):
        if frente is None and detras is not None:
            frente, detras = self._volcar(detras), None
        self._frente, self._detras = frente, detras

    @staticmethod
    def _volcar(detras):
        frente = None
        while detras is not None:
            x, detras = detras
            frente = (x, frente)
        return frente

    @classmethod
    def vacia(cls):
        return cls()

    def encolar(self, x):
        if self._frente is None:
            return type(self)((x, None))
        return type(self)(self._frente, (x, self._detras))

    def primero(self):
        if self._frente is None:
            raise IndexError("primero de una cola vacía")
        return self._frente[0]

    def resto(self):
        if self._frente is None:
            raise IndexError("resto de una cola vacía")
        return type(self)(self._frente[1], self._detras)

    def es_vacia(self):
        return self._frente is None


class ColaDeque[T](Cola[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=None):
        self._elementos = deque() if elementos is None else elementos

    @classmethod
    def vacia(cls):
        return cls()

    def encolar(self, x):
        d = self._elementos.copy()
        d.append(x)
        return type(self)(d)

    def primero(self):
        if not self._elementos:
            raise IndexError("primero de una cola vacía")
        return self._elementos[0]

    def resto(self):
        if not self._elementos:
            raise IndexError("resto de una cola vacía")
        d = self._elementos.copy()
        d.popleft()
        return type(self)(d)

    def es_vacia(self):
        return not self._elementos


IMPLEMENTACIONES_COLA = (ColaLista, ColaDosListas, ColaDeque)

c = ColaDosListas.vacia().encolar(3).encolar(2).encolar(5)
print(f"{c}   frente = {c._frente}   detras = {c._detras}")
c = c.resto()
print(f"{c}        frente = {c._frente}   detras = {c._detras}")
assert c == ColaDeque.vacia().encolar(2).encolar(5) == ColaLista([2, 5])

3 | 2 | 5   frente = (3, None)   detras = (5, (2, None))
2 | 5        frente = (2, (5, None))   detras = None


In [13]:
def colas(impl, elementos=st.integers(-50, 50), max_size=25):
    """Colas construidas con encolar (un entero) y resto (None) mezclados."""

    def construir(operaciones):
        c = impl.vacia()
        for op in operaciones:
            if op is not None:
                c = c.encolar(op)
            elif not c.es_vacia():
                c = c.resto()
        return c

    return st.lists(st.none() | elementos, max_size=max_size).map(construir)


for impl in IMPLEMENTACIONES_COLA:

    @given(colas(impl), st.integers())
    def axiomas_cola(c, x):
        v = impl.vacia()
        assert v.encolar(x).primero() == x and v.encolar(x).resto() == v
        if not c.es_vacia():
            assert c.encolar(x).primero() == c.primero()
            assert c.encolar(x).resto() == c.resto().encolar(x)
        assert v.es_vacia() and not c.encolar(x).es_vacia()

    assert comprobar(axiomas_cola, f"axiomas de {impl.__name__}") is None

axiomas de ColaLista: pasa


axiomas de ColaDosListas: pasa


axiomas de ColaDeque: pasa

## 11. Coste amortizado de la cola de dos listas

Contar cuántos elementos pasan de `detras` a `frente` con una subclase que sobrescriba `_volcar`. Comprobar que, encolando y sacando 10 000 elementos (primero todos seguidos y después mezclados al azar), cada elemento se vuelca como mucho una vez, así que el coste total es $O(n)$ y cada operación cuesta $O(1)$ amortizado. Después mostrar que esa cuenta solo vale si cada versión de la cola se usa una vez: pedir muchas veces `resto()` a la misma cola, con un solo elemento en `frente` y el resto en `detras`, repite el volcado entero cada vez.

In [14]:
class ColaDosListasContada(ColaDosListas):
    __slots__ = ()
    volcados = 0

    @staticmethod
    def _volcar(detras):
        frente = ColaDosListas._volcar(detras)
        n, f = 0, frente
        while f is not None:
            n, f = n + 1, f[1]
        ColaDosListasContada.volcados += n
        return frente


n = 10_000
c = ColaDosListasContada.vacia()
for x in range(n):
    c = c.encolar(x)
sacados = []
while not c.es_vacia():
    sacados.append(c.primero())
    c = c.resto()
print(f"Todos seguidos: {n} encolados, {ColaDosListasContada.volcados} volcados")
assert sacados == list(range(n)) and ColaDosListasContada.volcados <= n

random.seed(7)
ColaDosListasContada.volcados = 0
c, encolados, modelo = ColaDosListasContada.vacia(), 0, deque()
for _ in range(2 * n):
    if random.random() < 0.55 or c.es_vacia():
        c, encolados = c.encolar(encolados), encolados + 1
        modelo.append(encolados - 1)
    else:
        assert c.primero() == modelo.popleft()
        c = c.resto()
print(f"Mezclados     : {encolados} encolados, {ColaDosListasContada.volcados} volcados")
assert ColaDosListasContada.volcados <= encolados

c = ColaDosListasContada.vacia()
for x in range(1000):
    c = c.encolar(x)
ColaDosListasContada.volcados = 0
for _ in range(50):
    c.resto()
print(f"Reutilizando la misma versión: 50 × resto() vuelca {ColaDosListasContada.volcados} elementos")
assert ColaDosListasContada.volcados == 50 * 999

Todos seguidos: 10000 encolados, 9999 volcados
Mezclados     : 11114 encolados, 10129 volcados
Reutilizando la misma versión: 50 × resto() vuelca 49950 elementos


## 12. Conversión, último, longitud, todos y alguno

Escribir con la interfaz `lista_a_cola(xs, impl)`, `cola_a_lista(c)`, `ultimo_cola(c)` (el último en llegar; `ValueError` si está vacía), `longitud_cola(c)`, `todos_verifican(pred, c)` y `alguno_verifica(pred, c)`. Comprobarlas en las tres implementaciones contra las operaciones de las listas.

In [15]:
def lista_a_cola(xs, impl):
    return reduce(lambda c, x: c.encolar(x), xs, impl.vacia())


def cola_a_lista(c):
    return list(recorrer(c))


def ultimo_cola(c):
    if c.es_vacia():
        raise ValueError("último de una cola vacía")
    while not c.resto().es_vacia():
        c = c.resto()
    return c.primero()


def longitud_cola(c):
    n = 0
    while not c.es_vacia():
        n, c = n + 1, c.resto()
    return n


def todos_verifican(pred, c):
    while not c.es_vacia():
        if not pred(c.primero()):
            return False
        c = c.resto()
    return True


def alguno_verifica(pred, c):
    return not todos_verifican(lambda x: not pred(x), c)


c = lista_a_cola(["Ana", "Beto", "Carla", "Dácil"], ColaDosListas)
print(f"{c}: último {ultimo_cola(c)}, longitud {longitud_cola(c)}")
print("¿Todos con más de 2 letras?", todos_verifican(lambda s: len(s) > 2, c))
print("¿Alguno empieza por D?     ", alguno_verifica(lambda s: s.startswith("D"), c))

for impl in IMPLEMENTACIONES_COLA:

    @given(colas(impl))
    def derivadas_basicas(c):
        xs = cola_a_lista(c)
        assert lista_a_cola(xs, impl) == c
        assert longitud_cola(c) == len(xs)
        if xs:
            assert ultimo_cola(c) == xs[-1]
        assert todos_verifican(lambda x: x > 0, c) == all(x > 0 for x in xs)
        assert alguno_verifica(lambda x: x % 7 == 0, c) == any(x % 7 == 0 for x in xs)

    assert comprobar(derivadas_basicas, f"último, longitud, todos y alguno ({impl.__name__})") is None

Ana | Beto | Carla | Dácil: último Dácil, longitud 4
¿Todos con más de 2 letras? True
¿Alguno empieza por D?      True
último, longitud, todos y alguno (ColaLista): pasa


último, longitud, todos y alguno (ColaDosListas): pasa


último, longitud, todos y alguno (ColaDeque): pasa


## 13. Extender, intercalar y agrupar colas por turnos

Escribir `extender_cola(c1, c2)`, que encole en `c1` los elementos de `c2`; `intercalar_colas(c1, c2)`, que alterne un elemento de cada cola y añada al final lo que sobre de la más larga; y `agrupar_colas(colas, impl)`, que haga lo mismo con varias colas por turnos, usando a su vez una cola de colas: se saca una cola, se toma su primero y, si le quedan elementos, vuelve al final. Aplicarlo a tres taquillas del muelle de Los Cristianos y comprobarlo contra `itertools.zip_longest`.

In [16]:
def extender_cola(c1, c2):
    for x in recorrer(c2):
        c1 = c1.encolar(x)
    return c1


def intercalar_colas(c1, c2):
    r = c1.vacia()
    while not (c1.es_vacia() and c2.es_vacia()):
        if not c1.es_vacia():
            r, c1 = r.encolar(c1.primero()), c1.resto()
        if not c2.es_vacia():
            r, c2 = r.encolar(c2.primero()), c2.resto()
    return r


def agrupar_colas(colas, impl):
    turnos = lista_a_cola([c for c in colas if not c.es_vacia()], impl)
    r = impl.vacia()
    while not turnos.es_vacia():
        c, turnos = turnos.primero(), turnos.resto()
        r = r.encolar(c.primero())
        if not c.resto().es_vacia():
            turnos = turnos.encolar(c.resto())
    return r


def por_turnos(listas):
    return [x for grupo in zip_longest(*listas, fillvalue=_FALTA) for x in grupo if x is not _FALTA]


taquillas = [
    lista_a_cola(["Ana", "Beto", "Carla"], ColaLista),
    lista_a_cola(["Dácil", "Elio"], ColaDeque),
    lista_a_cola(["Fayna", "Guille", "Hugo", "Iballa"], ColaDosListas),
]
for i, t in enumerate(taquillas, 1):
    print(f"Taquilla {i}: {t}")
print("Embarque  :", agrupar_colas(taquillas, ColaDeque))
print("Extender 1 con 2  :", extender_cola(taquillas[0], taquillas[1]))
print("Intercalar 1 y 3  :", intercalar_colas(taquillas[0], taquillas[2]))
assert cola_a_lista(agrupar_colas(taquillas, ColaLista)) == por_turnos([cola_a_lista(t) for t in taquillas])

Taquilla 1: Ana | Beto | Carla
Taquilla 2: Dácil | Elio
Taquilla 3: Fayna | Guille | Hugo | Iballa
Embarque  : Ana | Dácil | Fayna | Beto | Elio | Guille | Carla | Hugo | Iballa
Extender 1 con 2  : Ana | Beto | Carla | Dácil | Elio
Intercalar 1 y 3  : Ana | Fayna | Beto | Guille | Carla | Hugo | Iballa


In [17]:
for impl in IMPLEMENTACIONES_COLA:

    @given(st.lists(colas(impl, max_size=8), max_size=4))
    def combinar_colas(cs):
        listas = [cola_a_lista(c) for c in cs]
        assert cola_a_lista(agrupar_colas(cs, impl)) == por_turnos(listas)
        if len(cs) >= 2:
            c1, c2 = cs[:2]
            assert cola_a_lista(extender_cola(c1, c2)) == listas[0] + listas[1]
            assert intercalar_colas(c1, c2) == agrupar_colas([c1, c2], impl)

    assert comprobar(combinar_colas, f"extender, intercalar y agrupar ({impl.__name__})") is None

extender, intercalar y agrupar (ColaLista): pasa


extender, intercalar y agrupar (ColaDosListas): pasa


extender, intercalar y agrupar (ColaDeque): pasa


## 14. Pertenencia, contención, prefijo, subcola, orden y máximo en colas

Escribir para colas, apoyándose en las funciones anteriores cuando se pueda: `pertenece_cola(x, c)`, `contenida_cola(c1, c2)`, `es_prefijo_cola(c1, c2)` (los elementos de `c1` son los primeros de `c2`), `es_subcola(c1, c2)` (aparecen seguidos en `c2`), `ordenada_cola(c)` (crecen del primero al último) y `maximo_cola(c)`. Comprobarlas contra las listas en las tres implementaciones, mezclando implementaciones en los argumentos.

In [18]:
def pertenece_cola(x, c):
    return alguno_verifica(lambda y: y == x, c)


def contenida_cola(c1, c2):
    return todos_verifican(lambda x: pertenece_cola(x, c2), c1)


def es_prefijo_cola(c1, c2):
    while not c1.es_vacia():
        if c2.es_vacia() or c1.primero() != c2.primero():
            return False
        c1, c2 = c1.resto(), c2.resto()
    return True


def es_subcola(c1, c2):
    while not es_prefijo_cola(c1, c2):
        if c2.es_vacia():
            return False
        c2 = c2.resto()
    return True


def ordenada_cola(c):
    while not c.es_vacia() and not c.resto().es_vacia():
        if c.primero() > c.resto().primero():
            return False
        c = c.resto()
    return True


def maximo_cola(c):
    if c.es_vacia():
        raise ValueError("máximo de una cola vacía")
    return reduce(max, recorrer(c))


c = lista_a_cola([2, 4, 4, 9, 3], ColaDosListas)
for xs in ([2, 4], [4, 9], [4, 3], [9, 3, 1]):
    q = lista_a_cola(xs, ColaLista)
    print(f"{str(q):10} contenida: {contenida_cola(q, c)!s:5}  prefijo: {es_prefijo_cola(q, c)!s:5}  subcola: {es_subcola(q, c)!s:5}  de {c}")
print(f"¿{c} ordenada? {ordenada_cola(c)};  máximo: {maximo_cola(c)}")

for impl, otra in zip(IMPLEMENTACIONES_COLA, IMPLEMENTACIONES_COLA[1:] + IMPLEMENTACIONES_COLA[:1]):

    @given(colas(impl, st.integers(0, 2), 6), colas(otra, st.integers(0, 2), 14), st.integers(0, 3))
    def consultas_colas(c1, c2, x):
        xs, ys = cola_a_lista(c1), cola_a_lista(c2)
        assert pertenece_cola(x, c2) == (x in ys)
        assert contenida_cola(c1, c2) == all(x in ys for x in xs)
        assert es_prefijo_cola(c1, c2) == (ys[: len(xs)] == xs)
        assert es_subcola(c1, c2) == es_sublista(xs, ys)
        assert ordenada_cola(c2) == (ys == sorted(ys))
        if ys:
            assert maximo_cola(c2) == max(ys)

    assert comprobar(consultas_colas, f"consultas ({impl.__name__} con {otra.__name__})") is None

2 | 4      contenida: True   prefijo: True   subcola: True   de 2 | 4 | 4 | 9 | 3
4 | 9      contenida: True   prefijo: False  subcola: True   de 2 | 4 | 4 | 9 | 3
4 | 3      contenida: True   prefijo: False  subcola: False  de 2 | 4 | 4 | 9 | 3
9 | 3 | 1  contenida: False  prefijo: False  subcola: False  de 2 | 4 | 4 | 9 | 3
¿2 | 4 | 4 | 9 | 3 ordenada? False;  máximo: 9


consultas (ColaLista con ColaDosListas): pasa
consultas (ColaDosListas con ColaDeque): pasa


consultas (ColaDeque con ColaLista): pasa


## 15. Cola de prioridad: lista ordenada frente a `heapq`

Definir el TAD `ColaPrioridad[T]` con `vacia()`, `insertar(x)`, `primero()` (el menor), `resto()` y `es_vacia()`, e implementarlo con una lista ordenada (insertando con `bisect.insort`) y con un montículo de `heapq`. Las dos copian la lista para ser persistentes; en una versión mutable la lista ordenada cuesta $O(n)$ al insertar y el montículo $O(\log n)$ tanto al insertar como al quitar el primero, que es por lo que se usa `heapq`. Atender por prioridad los avisos de un centro de emergencias (1 es lo más urgente) y escribir `ordenar_con(impl, xs)`, que ordene una lista metiendo todo en una cola de prioridad y sacándolo después. Comprobarlo contra `sorted`.

In [19]:
class ColaPrioridad[T](ABC):
    __slots__ = ()

    @classmethod
    @abstractmethod
    def vacia(cls) -> Self: ...

    @abstractmethod
    def insertar(self, x: T) -> Self: ...

    @abstractmethod
    def primero(self) -> T: ...

    @abstractmethod
    def resto(self) -> Self: ...

    @abstractmethod
    def es_vacia(self) -> bool: ...

    def __str__(self):
        return " | ".join(map(str, recorrer(self))) or "(vacía)"

    def __eq__(self, otra):
        if not isinstance(otra, ColaPrioridad):
            return NotImplemented
        return all(a == b for a, b in zip_longest(recorrer(self), recorrer(otra), fillvalue=_FALTA))


class ColaPrioridadLista[T](ColaPrioridad[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacia(cls):
        return cls()

    def insertar(self, x):
        xs = self._elementos.copy()
        bisect.insort(xs, x)
        return type(self)(xs)

    def primero(self):
        if not self._elementos:
            raise IndexError("primero de una cola de prioridad vacía")
        return self._elementos[0]

    def resto(self):
        if not self._elementos:
            raise IndexError("resto de una cola de prioridad vacía")
        return type(self)(self._elementos[1:])

    def es_vacia(self):
        return not self._elementos


class ColaPrioridadHeap[T](ColaPrioridad[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacia(cls):
        return cls()

    def insertar(self, x):
        h = self._elementos.copy()
        heapq.heappush(h, x)
        return type(self)(h)

    def primero(self):
        if not self._elementos:
            raise IndexError("primero de una cola de prioridad vacía")
        return self._elementos[0]

    def resto(self):
        if not self._elementos:
            raise IndexError("resto de una cola de prioridad vacía")
        h = self._elementos.copy()
        heapq.heappop(h)
        return type(self)(h)

    def es_vacia(self):
        return not self._elementos


IMPLEMENTACIONES_PRIORIDAD = (ColaPrioridadLista, ColaPrioridadHeap)


def ordenar_con(impl, xs):
    return list(recorrer(reduce(lambda c, x: c.insertar(x), xs, impl.vacia())))


avisos = [
    (3, "farola caída en Arrecife"),
    (1, "incendio forestal en La Palma"),
    (2, "accidente en la TF-5"),
    (1, "rescate en el Roque Nublo"),
    (3, "ruido en una terraza de Las Canteras"),
    (2, "inundación en un garaje de Telde"),
    (1, "parada cardíaca en Morro Jable"),
]
for impl in IMPLEMENTACIONES_PRIORIDAD:
    c = reduce(lambda c, a: c.insertar(a), avisos, impl.vacia())
    print(f"{impl.__name__:18} prioridades en su lista interna: {[p for p, _ in c._elementos]}")
for prioridad, aviso in ordenar_con(ColaPrioridadHeap, avisos):
    print(f"  {prioridad}  {aviso}")
assert ordenar_con(ColaPrioridadLista, avisos) == ordenar_con(ColaPrioridadHeap, avisos) == sorted(avisos)

for impl in IMPLEMENTACIONES_PRIORIDAD:

    @given(st.lists(st.integers(-20, 20), max_size=30))
    def ordena(xs):
        assert ordenar_con(impl, xs) == sorted(xs)

    assert comprobar(ordena, f"ordenar con {impl.__name__}") is None

ColaPrioridadLista prioridades en su lista interna: [1, 1, 1, 2, 2, 3, 3]
ColaPrioridadHeap  prioridades en su lista interna: [1, 1, 1, 3, 3, 2, 2]
  1  incendio forestal en La Palma
  1  parada cardíaca en Morro Jable
  1  rescate en el Roque Nublo
  2  accidente en la TF-5
  2  inundación en un garaje de Telde
  3  farola caída en Arrecife
  3  ruido en una terraza de Las Canteras


ordenar con ColaPrioridadLista: pasa
ordenar con ColaPrioridadHeap: pasa


## 16. Todas las implementaciones se comportan igual

Generar con Hypothesis secuencias aleatorias de operaciones (un entero para apilar, encolar o insertar; `None` para desapilar o sacar el primero si hay algo) y aplicarlas a la vez a todas las implementaciones de cada TAD y a un modelo hecho con una `list`. Después de cada operación, todas deben coincidir con el modelo en `es_vacia` y en sus elementos (sacados con la interfaz). Añadir una `ColaDosListasMal` que olvida el invariante en `resto` y ver el contraejemplo mínimo que encuentra Hypothesis.

In [20]:
operaciones = st.lists(st.none() | st.integers(-20, 20), max_size=40)


@given(operaciones)
def pilas_equivalentes(ops):
    ps, modelo = [impl.vacia() for impl in IMPLEMENTACIONES_PILA], []
    for op in ops:
        if op is not None:
            ps = [p.apilar(op) for p in ps]
            modelo.append(op)
        elif modelo:
            ps = [p.desapilar() for p in ps]
            modelo.pop()
        for p in ps:
            assert (p.es_vacia(), pila_a_lista(p)) == (not modelo, modelo)


def colas_equivalentes(implementaciones):
    @given(operaciones)
    def prueba(ops):
        cs, modelo = [impl.vacia() for impl in implementaciones], []
        for op in ops:
            if op is not None:
                cs = [c.encolar(op) for c in cs]
                modelo.append(op)
            elif modelo:
                cs = [c.resto() for c in cs]
                modelo.pop(0)
            for c in cs:
                assert (c.es_vacia(), cola_a_lista(c)) == (not modelo, modelo)

    return prueba


@given(operaciones)
def prioridades_equivalentes(ops):
    cs, modelo = [impl.vacia() for impl in IMPLEMENTACIONES_PRIORIDAD], []
    for op in ops:
        if op is not None:
            cs = [c.insertar(op) for c in cs]
            modelo.append(op)
        elif modelo:
            cs = [c.resto() for c in cs]
            modelo.remove(min(modelo))
        for c in cs:
            assert (c.es_vacia(), list(recorrer(c))) == (not modelo, sorted(modelo))


assert comprobar(pilas_equivalentes) is None
assert comprobar(colas_equivalentes(IMPLEMENTACIONES_COLA), "colas_equivalentes") is None
assert comprobar(prioridades_equivalentes) is None

pilas_equivalentes: pasa


colas_equivalentes: pasa
prioridades_equivalentes: pasa


In [21]:
class ColaDosListasMal(ColaDosListas):
    __slots__ = ()

    def resto(self):
        c = object.__new__(type(self))
        c._frente, c._detras = self._frente[1], self._detras
        return c


error = comprobar(colas_equivalentes(IMPLEMENTACIONES_COLA + (ColaDosListasMal,)), "colas con ColaDosListasMal")
assert error is not None and "ops=[0, 0]" in str(error.__notes__)
print("Al recorrer [0, 0], resto() deja frente vacía con un 0 en detras y la cola parece vacía antes de tiempo.")

colas con ColaDosListasMal: FALLA (AssertionError)
    Failing test case: prueba(
        ops=[0, 0],
    )
Al recorrer [0, 0], resto() deja frente vacía con un 0 en detras y la cola parece vacía antes de tiempo.
